In [1]:
# Change working directory
from pathlib import Path
import os

path = Path(r"P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\Beacon")

os.chdir(path)

print(Path.cwd())

P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\Beacon


In [2]:
# Load python libraries
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

In [3]:
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

# Load pre-project report
pre_rpt = read_rpt_file("SSF_SDMP_Beacon_pre.rpt")

# Extract data
pre = pre_rpt.node_flooding_summary.copy()

# Ensure Node is index
pre.index.name = "Node"

# Rename columns
pre = pre.rename(columns={
    "Hours_Flooded": "Pre Hours Flooded",
    "Maximum_Rate_CFS": "Pre Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Pre Total Flood Vol (MG)",
})

# Keep only needed columns
pre = pre[[
    "Pre Hours Flooded",
    "Pre Max Flood Rate (cfs)",
    "Pre Total Flood Vol (MG)"
]]

pre.head()

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG)
Node,,,
sQ2082,0.86,2.36,0.005
sQ2083,0.97,1.87,0.005
sQ2084,23.78,1.74,0.011
sQ2085,23.81,1.85,0.009
sQ2086,23.90,7.08,0.016


In [4]:
# Load post-project report
post_rpt = read_rpt_file("SSF_SDMP_Beacon_imp2.rpt")

# Extract data
post = post_rpt.node_flooding_summary.copy()

# Ensure Node is index
post.index.name = "Node"

# Rename columns
post = post.rename(columns={
    "Hours_Flooded": "Post Hours Flooded",
    "Maximum_Rate_CFS": "Post Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Post Total Flood Vol (MG)",
})

# Keep only needed columns
post = post[[
    "Post Hours Flooded",
    "Post Max Flood Rate (cfs)",
    "Post Total Flood Vol (MG)"
]]

post.head()

,Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG)
Node,,,
sQ2082,0.86,1.59,0.005
sQ2083,0.97,1.87,0.005
sQ2084,19.90,1.74,0.010
sQ2085,20.03,1.37,0.008
sQ2086,21.02,1.78,0.015


In [5]:
# Merge on index (Node)
comparison = pre.join(post, how="outer")

# Fill missing values
fill_cols = [
    "Pre Total Flood Vol (MG)",
    "Post Total Flood Vol (MG)",
    "Pre Hours Flooded",
    "Post Hours Flooded"
]

for col in fill_cols:
    comparison[col] = comparison[col].fillna(0)

# -----------------------------
# Reduction calculations
# -----------------------------

# Flood volume reduction
comparison["Total Flood Vol Reduction (MG)"] = (
    comparison["Pre Total Flood Vol (MG)"] - comparison["Post Total Flood Vol (MG)"]
)

comparison["Total Flood Vol Percent Reduction"] = np.where(
    comparison["Pre Total Flood Vol (MG)"] > 0,
    (comparison["Total Flood Vol Reduction (MG)"] / comparison["Pre Total Flood Vol (MG)"]) * 100,
    0
)

# Flood duration reduction
comparison["Hours Flooded Reduction"] = (
    comparison["Pre Hours Flooded"] - comparison["Post Hours Flooded"]
)

comparison["Hours Flooded Percent Reduction"] = np.where(
    comparison["Pre Hours Flooded"] > 0,
    (comparison["Hours Flooded Reduction"] / comparison["Pre Hours Flooded"]) * 100,
    0
)

# Round values
comparison["Total Flood Vol Reduction (MG)"] = comparison["Total Flood Vol Reduction (MG)"].round(3)
comparison["Total Flood Vol Percent Reduction"] = comparison["Total Flood Vol Percent Reduction"].round(1)

comparison["Hours Flooded Reduction"] = comparison["Hours Flooded Reduction"].round(2)
comparison["Hours Flooded Percent Reduction"] = comparison["Hours Flooded Percent Reduction"].round(1)

# Sort by severity
comparison = comparison.sort_values(
    by="Pre Total Flood Vol (MG)",
    ascending=False
)

comparison.head(20)

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG),Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG),Total Flood Vol Reduction (MG),Total Flood Vol Percent Reduction,Hours Flooded Reduction,Hours Flooded Percent Reduction
Node,,,,,,,,,,
sQ2094,23.95,12.45,0.016,21.53,1.26,0.015,0.001,6.3,2.42,10.1
sQ2086,23.90,7.08,0.016,21.02,1.78,0.015,0.001,6.3,2.88,12.1
sQ2087,23.88,3.80,0.015,20.62,3.22,0.014,0.001,6.7,3.26,13.7
sQ2093,23.96,6.02,0.014,21.68,1.12,0.014,0.000,0.0,2.28,9.5
sQ2088,23.90,3.44,0.013,20.88,1.62,0.013,0.000,0.0,3.02,12.6
sQ2095,23.96,8.18,0.013,21.42,1.01,0.012,0.001,7.7,2.54,10.6
sQ412,23.93,5.59,0.012,21.13,1.23,0.011,0.001,8.3,2.80,11.7
sQ2084,23.78,1.74,0.011,19.90,1.74,0.010,0.001,9.1,3.88,16.3
sQ2092A,23.94,2.94,0.011,21.15,1.06,0.010,0.001,9.1,2.79,11.7


In [6]:
# Reset index so Node becomes a column
export_df = comparison.reset_index()

# Export to CSV
export_df.to_csv("flood_comparison.csv", index=False)

print("Exported: flood_comparison.csv")

Exported: flood_comparison.csv


In [7]:
# -----------------------------
# Entire network flood reduction
# -----------------------------

# --- Volume ---
network_pre_total = comparison["Pre Total Flood Vol (MG)"].sum()
network_post_total = comparison["Post Total Flood Vol (MG)"].sum()
network_reduction_mg = network_pre_total - network_post_total

network_percent_reduction = np.where(
    network_pre_total > 0,
    (network_reduction_mg / network_pre_total) * 100,
    0
)

# --- Duration ---
network_pre_hours = comparison["Pre Hours Flooded"].sum()
network_post_hours = comparison["Post Hours Flooded"].sum()
network_hours_reduction = network_pre_hours - network_post_hours

network_hours_percent_reduction = np.where(
    network_pre_hours > 0,
    (network_hours_reduction / network_pre_hours) * 100,
    0
)

# -----------------------------
# Print results
# -----------------------------

print(f"Pre-Project Network Total Flood Volume: {network_pre_total:.3f} MG")
print(f"Post-Project Network Total Flood Volume: {network_post_total:.3f} MG")
print(f"Network Flood Volume Reduction: {network_reduction_mg:.3f} MG")
print(f"Network Flood Volume Percent Reduction: {network_percent_reduction:.1f}%")

print("\n--- Flood Duration ---")
print(f"Pre-Project Total Hours Flooded: {network_pre_hours:.2f} hrs")
print(f"Post-Project Total Hours Flooded: {network_post_hours:.2f} hrs")
print(f"Total Hours Flooded Reduction: {network_hours_reduction:.2f} hrs")
print(f"Hours Flooded Percent Reduction: {network_hours_percent_reduction:.1f}%")

Pre-Project Network Total Flood Volume: 0.187 MG
Post-Project Network Total Flood Volume: 0.172 MG
Network Flood Volume Reduction: 0.015 MG
Network Flood Volume Percent Reduction: 8.0%

--- Flood Duration ---
Pre-Project Total Hours Flooded: 407.61 hrs
Post-Project Total Hours Flooded: 351.54 hrs
Total Hours Flooded Reduction: 56.07 hrs
Hours Flooded Percent Reduction: 13.8%


In [8]:
# Fix: ensure these are scalars (not numpy arrays)
network_percent_reduction = float(network_percent_reduction)
network_hours_percent_reduction = float(network_hours_percent_reduction)

# -----------------------------
# Create summary table
# -----------------------------
summary_df = pd.DataFrame({
    "Description": [
        "Pre-Project Network Total Flood Volume (MG)",
        "Post-Project Network Total Flood Volume (MG)",
        "Network Flood Volume Reduction (MG)",
        "Network Flood Volume Percent Reduction (%)",
        "Pre-Project Total Hours Flooded (hrs)",
        "Post-Project Total Hours Flooded (hrs)",
        "Total Hours Flooded Reduction (hrs)",
        "Hours Flooded Percent Reduction (%)"
    ],
    "Value": [
        round(network_pre_total, 3),
        round(network_post_total, 3),
        round(network_reduction_mg, 3),
        f"{round(network_percent_reduction, 1)}%",
        round(network_pre_hours, 2),
        round(network_post_hours, 2),
        round(network_hours_reduction, 2),
        f"{round(network_hours_percent_reduction, 1)}%"
    ]
})

# -----------------------------
# Export summary CSV
# -----------------------------
summary_df.to_csv("network_flood_summary.csv", index=False)

print("Exported: network_flood_summary.csv")

Exported: network_flood_summary.csv
